[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/mrseq/blob/main/examples/epi3d.ipynb)

# 3D Echo Planar Imaging

Qualitative 3D imaging with a segmented 3D EPI readout. Each partition (kz encoding step) is acquired after a separate slab-selective excitation.

### Imports

In [ ]:
import importlib

if not importlib.util.find_spec('mrseq'):
    %pip install mrseq[examples]

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import nibabel
import numpy as np
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryIsmrmrd
from raw2ismrmrd.utils import combine_ismrmrd_files
from scipy.ndimage import binary_erosion

from mrseq.sequences.epi3d_fid import main as create_seq_fid
from mrseq.sequences.epi3d_se import main as create_seq_se
from mrseq.utils import sys_defaults

### Settings
We use a 3D BrainWeb phantom with an in-plane matrix size of 64 x 64 and 8 partitions covering a ~60 mm slab.

In [ ]:
image_matrix_size = [64, 64]
n_partitions = 8
slab_thickness_mm = 60.0  # phantom slab thickness covered by the partitions [mm]
tr = 5.0  # long TR so the magnetization recovers between partition excitations

tmp = tempfile.TemporaryDirectory()

### Create the digital 3D phantom

We use a real 3D BrainWeb phantom generated by MRzero's `generate_brainweb_phantoms`. The phantom data is
downloaded once and cached in `output/brainweb` (which is gitignored), so subsequent runs are fast.

The full-resolution phantom (362 x 434 x 362 voxels at 0.5 mm) is resliced to the target grid *while
loading* via the phantom config's `reslice_to`, which averages each output voxel over the source region
it covers (density-weighted for intensive maps like T1/T2). Loading the full resolution and
interpolating afterwards would need more than 15 GB of RAM — reslicing during load keeps the memory
footprint small and also provides better partial-volume handling. The target grid is a slab of
`n_partitions` partitions around the center with isotropic in-plane voxels; the scalp fat tissue is
removed so the phantom contains only brain tissue (no fat ring or skull gap).

The acquisition FOV is square but the phantom is not (181 x 217 mm in-plane), so the phantom is
resampled to isotropic in-plane voxels matching the reconstruction voxel size and later zero-padded
in x for the comparison.

Since each partition is acquired after a separate 90° excitation, we use a long repetition time
(`tr = 5 s`) so the magnetization recovers between excitations — the sequence length does not affect
the MRzero simulation cost.

In [ ]:
brainweb_subj = Path.cwd() / 'output' / 'brainweb' / 'brainweb-subj04'
json_path = brainweb_subj / 'brainweb-subj04-3T.json'
if not json_path.exists():
    mr0.generate_brainweb_phantoms(str(brainweb_subj.parent), subject_count=1)

# source grid from the NIfTI header (cheap — no data is loaded)
src = nibabel.load(str(brainweb_subj / 'brainweb-subj04.nii.gz'))
src_affine = np.asarray(src.affine, dtype=float)  # voxel index -> scanner coordinates [mm]
src_shape = np.array(src.shape[:3])
src_voxel_size = np.abs(np.diag(src_affine)[:3])

# target grid: central slab of `n_partitions` partitions, isotropic in-plane voxels over the
# full phantom FOV (~53 x 64 x 8 samples over 181 x 217 x 60 mm)
n_x = round(image_matrix_size[0] * src_shape[0] * src_voxel_size[0] / (src_shape[1] * src_voxel_size[1]))
n_target = np.array([n_x, image_matrix_size[1], n_partitions])
fov_target = np.array([src_shape[0] * src_voxel_size[0], src_shape[1] * src_voxel_size[1], slab_thickness_mm])
scale = fov_target / (src_voxel_size * n_target)  # target voxel size in units of source voxels
offset = (src_shape - 1) / 2 - scale * (n_target - 1) / 2  # center the target grid on the source
grid = np.eye(4)
grid[:3, :3] = np.diag(scale)
grid[:3, 3] = offset
target_affine = src_affine @ grid

# reslice to the target grid while loading and combine the tissues into a single phantom
config = mr0.NiftiPhantom.load(str(json_path))
config.tissues.pop('fat')  # remove scalp fat so only brain tissue remains
config.reslice_to = mr0.ResliceTo(affine=target_affine.tolist(), resolution=n_target.tolist())
phantom = mr0.TissueDict.load(str(brainweb_subj), config).combine()
phantom.B1[:] = 1.0

fov_xy = float(phantom.size[:2].max())  # larger in-plane extent to avoid fold-over
fov_z = float(phantom.size[2])
phantom.size

### 3D EPI FID sequence

The 3D EPI FID sequence acquires one kz partition per slab-selective excitation. The slice-selection
rewinder gradient is merged with the kz pre-phaser (or, when navigator acquisitions are enabled, the kz
pre-phaser is played between the navigators and the readout while the navigators stay at kz = 0).

In [ ]:
sequence, fname_seq = create_seq_fid(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    tr=tr,
    fov_xy=fov_xy,
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[1],
    n_partitions=n_partitions,
    fov_z=fov_z,
    partial_fourier_factor=0.75,
    ramp_sampling=True,
    readout_type='symmetric',
)

### Simulation and reconstruction

In [ ]:
def simulate_and_reconstruct(fname_seq, sequence, name):
    """Simulate the sequence with MRzero and reconstruct with mrpro."""
    mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
    signal, _ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e-3)
    fname_mrd = Path(tmp.name) / f'{name}.mrd'
    mr0.sig_to_mrd(fname_mrd, signal, sequence)
    combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

    kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
    recon = DirectReconstruction(kdata, csm=None)
    return recon(kdata)

In [ ]:
idata = simulate_and_reconstruct(fname_seq, sequence, 'epi3d_fid')

In [ ]:
from copy import deepcopy

idat = idata.data.abs().squeeze().numpy()  # (z, y, x)

# common intensity scale for all partitions
vmax = np.sort(idat.flatten())[int(idat.size * 0.99)]

ref = deepcopy(phantom.PD.numpy()).transpose(2, 1, 0)  # (z, y, n_x)
t1 = deepcopy(phantom.T1.numpy()).transpose(2, 1, 0)

# zero-pad x symmetrically to the square recon matrix (FOV is square, phantom is not)
pad = image_matrix_size[0] - ref.shape[-1]
ref = np.pad(ref, ((0, 0), (0, 0), (pad // 2, pad - pad // 2)))

fig, ax = plt.subplots(2, n_partitions, figsize=(2 * n_partitions, 2.5))
for p in range(n_partitions):
    ax[0, p].imshow(idat[p], cmap='gray', vmin=0, vmax=vmax)
    ax[0, p].set_title(f'partition {p}')
    ax[0, p].set_xticks([])
    ax[0, p].set_yticks([])

    ax[1, p].imshow(ref[p], cmap='gray', vmin=0)
    ax[1, p].set_xticks([])
    ax[1, p].set_yticks([])
fig.tight_layout()
plt.show()

In [ ]:
# compare each reconstructed partition to the corresponding phantom slice
# (phantom maps are (x, y, z), recon is (z, y, x) -> transpose the reference)
# the sim produces a T1-weighted signal: |recon| ~ PD * (1 - exp(-TR / T1))


# z-localization: the partition signal profile must follow the phantom.
# The outermost partitions show slab-edge artifacts (spins at the slab boundary are only
# partially excited/refocused, which MRzero >= 1.5 models more accurately), so the check
# is strict on the inner partitions and looser on the full profile.
recon_profile = idat.mean(axis=(1, 2))
ref_profile = ref.mean(axis=(1, 2))
print('recon z profile:', recon_profile / recon_profile.max())
print('phantom z profile:', ref_profile / ref_profile.max())
corr_all = np.corrcoef(recon_profile, ref_profile)[0, 1]
corr_inner = np.corrcoef(recon_profile[1:-1], ref_profile[1:-1])[0, 1]
print(f'z-profile correlation: all = {corr_all:.3f}, inner = {corr_inner:.3f}')
assert corr_inner > 0.9
assert corr_all > 0.8

# per-partition relative error inside the brain (excluding the 2-voxel edge rim
# where partial-volume blurring dominates at this resolution)
relative_error = 0.0
for p in range(n_partitions):
    im = idat[p] / np.percentile(idat[p], 99)
    re = ref[p] / np.percentile(ref[p], 99)
    mask = binary_erosion(re > 0.05, iterations=2)
    relative_error += np.sum(np.abs(im - re)[mask]) / np.sum(np.abs(re)[mask])
relative_error /= n_partitions

print(f'Relative error {relative_error}')
assert relative_error < 0.3

### 3D EPI FID with partial Fourier along kz

Since each kz partition requires its own slab-selective excitation, partial Fourier along kz directly
reduces the number of excitations and hence the scan time (unlike partial Fourier along ky, which
shortens TE). Missing partitions are zero-filled in the reconstruction, which blurs the image along z.

In [ ]:
sequence_pfz, fname_seq_pfz = create_seq_fid(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    tr=tr,
    fov_xy=fov_xy,
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[1],
    n_partitions=n_partitions,
    fov_z=fov_z,
    partial_fourier_factor=0.75,
    partial_fourier_factor_z=0.75,
    ramp_sampling=True,
    readout_type='symmetric',
)
idata_pfz = simulate_and_reconstruct(fname_seq_pfz, sequence_pfz, 'epi3d_fid_pfz')
idat_pfz = idata_pfz.data.abs().squeeze().numpy()  # (z, y, x)
idat_pfz.shape

### 3D EPI SE

The 3D EPI spin echo sequence adds a slab-selective 180° refocusing pulse. The kz pre-phaser is played
with inverted sign before the refocusing pulse, so the readout afterwards samples the intended
+kz position.

In [ ]:
sequence_se, fname_seq_se = create_seq_se(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    tr=tr,
    fov_xy=fov_xy,
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[1],
    n_partitions=n_partitions,
    fov_z=fov_z,
    partial_fourier_factor=0.75,
    ramp_sampling=True,
    readout_type='symmetric',
)
idata_se = simulate_and_reconstruct(fname_seq_se, sequence_se, 'epi3d_se')
idat_se = idata_se.data.abs().squeeze().numpy()  # (z, y, x)
idat_se.shape

In [ ]:
def relative_error(idat, ref):
    """Brain-interior relative error vs. the reference (2-voxel edge rim excluded)."""
    err = 0.0
    for p in range(n_partitions):
        im = idat[p] / np.percentile(idat[p], 99)
        re = ref[p] / np.percentile(ref[p], 99)
        mask = binary_erosion(re > 0.05, iterations=2)
        err += np.sum(np.abs(im - re)[mask]) / np.sum(np.abs(re)[mask])
    return err / n_partitions


recons = [('FID', idat), ('FID pfz=0.75', idat_pfz), ('SE', idat_se)]
fig, ax = plt.subplots(3, n_partitions, figsize=(2 * n_partitions, 7))
for r, (name, img) in enumerate(recons):
    vmax = np.percentile(img, 99)
    for p in range(n_partitions):
        ax[r, p].imshow(img[p], cmap='gray', vmin=0, vmax=vmax)
        ax[r, p].set_xticks([])
        ax[r, p].set_yticks([])
    ax[r, 0].set_ylabel(name)
fig.tight_layout()
plt.show()

err_fid = relative_error(idat, ref)
err_pfz = relative_error(idat_pfz, ref)
err_se = relative_error(idat_se, ref)
print(f'Relative error FID      : {err_fid:.4f}')
print(f'Relative error FID pfz  : {err_pfz:.4f}')
print(f'Relative error SE       : {err_se:.4f}')
assert err_fid < 0.3
assert err_pfz < 0.35
assert err_se < 0.3